## 🔧 Environment Setup

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1 — ENVIRONMENT SETUP
# Install libraries, pull latest pipeline code from GitHub,
# create output folder structure, define session config.
# Run this once at the start of every weekly session.
# ═══════════════════════════════════════════════════════════
import os, sys, subprocess, shutil
from datetime import date

# Detect if running on Kaggle
IS_KAGGLE = os.path.exists('/kaggle')

# ── User config ─────────────────────────────────────────────
GITHUB_REPO    = "https://github.com/Iceandlava124/tess-exoplanet-pipeline.git"
KAGGLE_DATASET = "bhavishmehta/tess-exoplanet-discovery-results"
SESSION_LABEL  = date.today().strftime("%Y-%m-%d")
TIME_LIMIT_HRS = 5.0
DISK_LIMIT_GB  = 18
STARS_PER_RUN  = 400

if IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"
    PIPELINE_DIR = os.path.join(WORKING_DIR, "pipeline")
    RESULTS_DIR  = os.path.join(WORKING_DIR, "results")
else:
    WORKING_DIR = os.path.abspath(".")
    PIPELINE_DIR = WORKING_DIR
    RESULTS_DIR  = os.path.join(WORKING_DIR, "results")

# ── Install libraries quietly ────────────────────────────────
if IS_KAGGLE:
    print("Installing libraries...")
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q",
             "lightkurve", "wotan", "astropy", "astroquery",
             "batman-package", "transitleastsquares", "ldtk", "scipy", "scikit-learn",
             "imbalanced-learn", "tqdm", "joblib",
             "pandas", "numpy"],
            check=True, capture_output=True
        )
        print("Libraries installed.")
    except Exception as e:
        print(f"Library install warning (may already be installed): {e}")
else:
    print("Running locally. Skipping pip install cell (please run requirements.txt manually if needed).")

# ── Pull latest pipeline code from GitHub ───────────────────
if IS_KAGGLE:
    try:
        if os.path.isdir(PIPELINE_DIR):
            print("Updating existing pipeline clone...")
            result = subprocess.run(
                ["git", "-C", PIPELINE_DIR, "pull"],
                capture_output=True, text=True, timeout=120
            )
            print(result.stdout.strip() or "Already up to date.")
        else:
            print("Cloning pipeline repository...")
            result = subprocess.run(
                ["git", "clone", "--depth=1", GITHUB_REPO, PIPELINE_DIR],
                capture_output=True, text=True, timeout=180
            )
            if result.returncode != 0:
                raise RuntimeError(result.stderr.strip())
            print("Pipeline cloned successfully.")
    except Exception as e:
        print(f"❌ Git operation failed: {e}")
        print("   Continuing — if pipeline/ already exists the run may still work.")
else:
    print("Running locally. Using current directory codebase.")

# ── Add pipeline to Python path ──────────────────────────────
for p in [PIPELINE_DIR, WORKING_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

# ── Create output folder structure ──────────────────────────
for folder in [
    os.path.join(RESULTS_DIR, "KEEP"),
    os.path.join(RESULTS_DIR, "FLAG"),
    os.path.join(RESULTS_DIR, "DISCARD"),
    os.path.join(RESULTS_DIR, "plots", "flag_analysis"),
    os.path.join(RESULTS_DIR, "reports"),
    os.path.join(RESULTS_DIR, "figures"),
]:
    os.makedirs(folder, exist_ok=True)

print(f"✅ Environment ready — Session: {SESSION_LABEL} (Local: {not IS_KAGGLE})")

## 📂 Resume From Last Week

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2 — RESUME FROM LAST WEEK
# If previous results exist, skip already processed stars.
# ═══════════════════════════════════════════════════════════
import pandas as pd
from collections import Counter

if 'IS_KAGGLE' not in globals():
    IS_KAGGLE = os.path.exists('/kaggle')

if IS_KAGGLE:
    INPUT_RESULTS_CSV = f"/kaggle/input/tess-exoplanet-discovery-results/results.csv"
    INPUT_CANDIDATES  = f"/kaggle/input/tess-exoplanet-discovery-results/candidates_submission.csv"
else:
    INPUT_RESULTS_CSV = os.path.join(RESULTS_DIR, "results.csv")
    INPUT_CANDIDATES  = os.path.join(RESULTS_DIR, "candidates_submission.csv")

OUTPUT_RESULTS_CSV = os.path.join(RESULTS_DIR, "results.csv")
already_done = set()
df_previous  = None

try:
    if os.path.exists(INPUT_RESULTS_CSV):
        if IS_KAGGLE:
            shutil.copy2(INPUT_RESULTS_CSV, OUTPUT_RESULTS_CSV)
        df_previous = pd.read_csv(INPUT_RESULTS_CSV)
        if "tic_id" in df_previous.columns:
            already_done = set(df_previous["tic_id"].astype(int).tolist())
        print(f"📂 Loaded {len(df_previous)} previous results.")
        print(f"   Already processed: {len(already_done)} unique stars")
        if "decision" in df_previous.columns:
            counts = Counter(df_previous["decision"].fillna("UNKNOWN"))
            print(f"   KEEP:    {counts.get('KEEP', 0)}")
            print(f"   FLAG:    {counts.get('FLAG', 0)}")
            print(f"   DISCARD: {counts.get('DISCARD', 0)}")
    else:
        print("🆕 Fresh start — no previous results found")
except Exception as e:
    print(f"❌ Error loading previous results: {e}")

try:
    if os.path.exists(INPUT_CANDIDATES) and IS_KAGGLE:
        shutil.copy2(INPUT_CANDIDATES, os.path.join(RESULTS_DIR, "candidates_submission.csv"))
        print("   Candidates file copied.")
except Exception as e:
    print(f"   Warning: could not copy candidates file: {e}")

print(f"✅ Resume check complete — {len(already_done)} stars will be skipped.")

## 🎯 This Week's Targets

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3 — BUILD TARGET LIST
# ═══════════════════════════════════════════════════════════
import sys, os
if 'IS_KAGGLE' not in globals():
    IS_KAGGLE = os.path.exists('/kaggle')

for path_dir in ["/kaggle/working/pipeline", "/kaggle/working", os.path.abspath(".")]:
    if path_dir not in sys.path:
        sys.path.insert(0, path_dir)

try:
    from kaggle_discovery_runner import build_target_list
except Exception as e_imp:
    print(f"❌ Failed to import build_target_list: {e_imp}")
    print("Python Path is:", sys.path)
    raise e_imp

TARGETS_CSV = os.path.join(RESULTS_DIR, "this_week_targets.csv")

try:
    targets = build_target_list(
        n_targets            = STARS_PER_RUN,
        already_processed    = already_done if 'already_done' in globals() else set(),
        mag_range            = (8, 13),
        teff_range           = (3500, 7000),
        radius_range         = (0.5, 2.0),
        exclude_giants       = True,
        exclude_known_contaminated = True,
        prioritise_multi_sector    = True,
        prioritise_not_in_toi      = True,
    )
    targets.to_csv(TARGETS_CSV, index=False)
    n = len(targets)
    est_hrs = n * 45 / 3600
    print(f"🎯 {n} stars queued for this session")
    print(f"   Estimated time: {est_hrs:.1f} hours at ~45s per star")
    print(f"   Target list saved to: {TARGETS_CSV}")
except Exception as e:
    print(f"❌ Target list build failed: {e}")
    import traceback; traceback.print_exc()
    targets = pd.DataFrame(columns=["tic_id"])

print(f"✅ Target list ready — {len(targets)} stars")

## 🔭 Autonomous Discovery — ~8.5 hrs

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4 — AUTONOMOUS DISCOVERY
# This is the main cell. It runs the full 13-step pipeline
# on every star in the target list. It stops automatically
# when time or disk limits are hit. One bad star never
# crashes the entire run.
# Expected runtime: 6-9 hours on Kaggle GPU.
# ═══════════════════════════════════════════════════════════
try:
    from kaggle_discovery_runner import run_discovery_session
except ImportError:
    sys.path.insert(0, WORKING_DIR)
    from kaggle_discovery_runner import run_discovery_session

session_summary = {}
if len(targets) == 0:
    print("❌ No targets to process — re-run Cell 3 first.")
else:
    try:
        session_summary = run_discovery_session(
            targets           = targets,
            output_dir        = RESULTS_DIR,
            models_dir        = os.path.join(WORKING_DIR, "models"),
            time_limit_hours  = TIME_LIMIT_HRS,
            disk_limit_gb     = DISK_LIMIT_GB,
            save_every_n      = 50,
            session_label     = SESSION_LABEL,
            # Pipeline feature flags
            run_flag_analyzer      = True,
            run_candidate_export   = True,
            run_toi_crosscheck     = True,
            alias_rejection        = True,
            # Physical plausibility filters
            max_planet_radius_earth = 25.0,
            min_transit_snr         = 5.0,
            min_depth_ppm           = 100,
            # Log file
            log_file = os.path.join(RESULTS_DIR, "discovery_log.txt"),
        )
        n_proc    = session_summary.get("stars_processed", 0)
        n_skip    = session_summary.get("stars_skipped", 0)
        elapsed   = session_summary.get("elapsed_hours", 0)
        print(f"✅ Discovery session complete")
        print(f"   Stars processed: {n_proc}")
        print(f"   Stars skipped:   {n_skip}")
        print(f"   Time elapsed:    {elapsed:.1f} hrs")
    except Exception as e:
        print(f"❌ Discovery session crashed: {e}")
        import traceback; traceback.print_exc()
        session_summary = {}

## 📊 Session Summary

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5 — SESSION SUMMARY
# Print the final statistics table, highlight any potential
# new planet discoveries not in the TOI catalog, and append
# one line to the cumulative DISCOVERY_LOG.md.
# ═══════════════════════════════════════════════════════════
try:
    from kaggle_discovery_runner import generate_session_summary
except ImportError:
    sys.path.insert(0, WORKING_DIR)
    from kaggle_discovery_runner import generate_session_summary

try:
    summary = generate_session_summary(
        results_dir   = RESULTS_DIR,
        session_label = SESSION_LABEL,
        session_data  = session_summary,
    )

    n_attempted  = summary.get("stars_attempted", 0)
    n_processed  = summary.get("stars_processed", 0)
    n_skipped    = summary.get("stars_skipped", 0)
    n_alias      = summary.get("alias_discards", 0)
    elapsed      = summary.get("elapsed_hours", 0)
    throughput   = n_processed / elapsed if elapsed > 0 else 0
    n_keep       = summary.get("keep", 0)
    n_flag       = summary.get("flag", 0)
    n_discard    = summary.get("discard", 0)
    n_new        = summary.get("new_discoveries", 0)
    n_cumulative = summary.get("cumulative_total", 0)

    print("=" * 50)
    print(f"  TESS DISCOVERY RUN -- {SESSION_LABEL}")
    print("=" * 50)
    print(f"  Stars attempted:   {n_attempted}")
    print(f"  Stars processed:   {n_processed}")
    print(f"  Skipped (no data): {n_skipped}")
    print(f"  Alias discards:    {n_alias}")
    print(f"  Time elapsed:      {elapsed:.1f} hrs")
    print(f"  Throughput:        {throughput:.0f} stars/hr")
    print("-" * 50)
    print(f"  KEEP:    {n_keep} ({n_new} potential new discoveries)")
    print(f"  FLAG:    {n_flag} (human review needed)")
    print(f"  DISCARD: {n_discard}")
    print("-" * 50)
    print(f"  Cumulative total:  {n_cumulative} stars all-time")
    print("=" * 50)
except Exception as e:
    print(f"❌ Summary generation failed: {e}")
    summary = {}
    n_new = 0
    n_cumulative = 0

# Print any potential new discoveries not in the TOI catalog.
try:
    new_disc_file = os.path.join(RESULTS_DIR, "new_discoveries.txt")
    if os.path.exists(new_disc_file) and os.path.getsize(new_disc_file) > 0:
        with open(new_disc_file, "r") as f:
            lines = [l.strip() for l in f.readlines() if l.strip()]
        if lines:
            print("")
            print("POTENTIAL NEW DISCOVERIES NOT IN TOI CATALOG:")
            for line in lines:
                print(f"  {line}")
except Exception as e:
    print(f"   Warning: could not read new_discoveries.txt: {e}")

# Append one row to the cumulative discovery log markdown table.
try:
    log_path = os.path.join(RESULTS_DIR, "DISCOVERY_LOG.md")
    write_header = not os.path.exists(log_path)
    with open(log_path, "a", encoding="utf-8") as f:
        if write_header:
            f.write("# TESS Discovery Log\n\n")
            f.write("| Date | Processed | KEEP | FLAG | DISCARD | New | Notes |\n")
            f.write("|------|-----------|------|------|---------|-----|-------|\n")
        notes = f"Alias discards: {summary.get('alias_discards', 0)}"
        f.write(f"| {SESSION_LABEL} | {n_processed} | {summary.get('keep',0)} | "
                f"{summary.get('flag',0)} | {summary.get('discard',0)} | "
                f"{summary.get('new_discoveries',0)} | {notes} |\n")
    print("✅ Discovery log updated")
except Exception as e:
    print(f"❌ Log update failed: {e}")

## 💾 Save & Export

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 6 — SAVE & EXPORT
# ═══════════════════════════════════════════════════════════
import subprocess
if 'IS_KAGGLE' not in globals():
    IS_KAGGLE = os.path.exists('/kaggle')

export_map = {
    os.path.join(RESULTS_DIR, "results.csv"):                   "results_cumulative.csv",
    os.path.join(RESULTS_DIR, "candidates_submission.csv"):     "candidates_all.csv",
    os.path.join(RESULTS_DIR, "manual_review_queue.csv"):       "review_queue_latest.csv",
    os.path.join(RESULTS_DIR, "DISCOVERY_LOG.md"):              "DISCOVERY_LOG.md",
    os.path.join(RESULTS_DIR, "new_discoveries.txt"):           "new_discoveries.txt",
}

for src, dest_name in export_map.items():
    try:
        if os.path.exists(src) and os.path.getsize(src) > 0:
            dest = os.path.join(WORKING_DIR, dest_name)
            shutil.copy2(src, dest)
            print(f"Saved: {dest_name}")
        else:
            print(f"   Skipped (empty or missing): {dest_name}")
    except Exception as e:
        print(f"   Failed to copy {dest_name}: {e}")

if IS_KAGGLE:
    try:
        n_processed_final = session_summary.get("stars_processed", 0)
        commit_msg = f"Auto-update: {SESSION_LABEL} -- {n_processed_final} stars processed"
        print(f"Pushing results to Kaggle dataset '{KAGGLE_DATASET}'...")
        result = subprocess.run(
            ["kaggle", "datasets", "version",
             "-p", RESULTS_DIR,
             "-m", commit_msg,
             "--dir-mode", "zip"],
            capture_output=True, text=True, timeout=300
        )
        if result.returncode == 0:
            print("Dataset updated successfully.")
        else:
            print(f"   Kaggle push warning: {result.stderr.strip()}")
    except Exception as e:
        print(f"   Kaggle push failed (results still saved locally): {e}")
else:
    print("Running locally. Skipping Kaggle dataset upload.")

try:
    n_new_final  = summary.get("new_discoveries", 0)
    n_cumulative = summary.get("cumulative_total", 0)
except Exception:
    n_new_final  = 0
    n_cumulative = 0

print(f"\nAll done for {SESSION_LABEL}!")
print(f"   New discoveries: {n_new_final}")
print(f"   Cumulative stars analysed: {n_cumulative}")
print("✅ Export complete")